# DashBoard Analysis ANAYA Gelar Rasa 2026

In [1]:
import glob
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DASHBOARD = 'extracted_data' if os.path.exists('extracted_data') else '.'
print('Direktori data aktif:', DASHBOARD)

Direktori data aktif: .


## Tahap 1: Data Master & Rekayasa Atribut Wilayah
- **Tujuan 1 (Kondisi Kualitas Udara Ruang & Waktu):** Menurunkan kolom `pulau` dari `provinsi` untuk perbandingan antar 5 pulau besar.
- **Tujuan 2 & 3 (Waktu Kritis & Faktor Pemicu):** Mengelompokkan 19 string unik `site_type` menjadi **5 Kelas Kawasan Utama**.
- **Tujuan 4 (Dampak Kesehatan):** Menyusun kunci relasi eksplisit `SENS-0nn <-> RS-XXX-nn` agar 3 stasiun DKI Jakarta tidak terbuang saat *join*.
- **Dashboard:** Menyediakan dimensi filter global lintas Halaman 1, 2, dan 3.

In [2]:
sites = pd.read_csv('sites.csv')
sensors = pd.read_csv('sensors.csv')
hospitals = pd.read_csv('hospitals.csv')
health = pd.read_csv('HLT-01_health_outcomes_daily.csv')

# Tujuan 1: Mapping Provinsi -> Pulau
provinsi_to_pulau = {
    'Aceh': 'Sumatera', 'Sumatera Utara': 'Sumatera', 'Sumatera Barat': 'Sumatera',
    'Riau': 'Sumatera', 'Kepulauan Riau': 'Sumatera', 'Jambi': 'Sumatera',
    'Sumatera Selatan': 'Sumatera', 'Bengkulu': 'Sumatera', 'Lampung': 'Sumatera',
    'DKI Jakarta': 'Jawa', 'Jawa Barat': 'Jawa', 'D.I. Yogyakarta': 'Jawa', 'Jawa Timur': 'Jawa',
    'Kalimantan Barat': 'Kalimantan', 'Kalimantan Tengah': 'Kalimantan',
    'Kalimantan Selatan': 'Kalimantan', 'Kalimantan Timur': 'Kalimantan',
    'Sulawesi Selatan': 'Sulawesi', 'Sulawesi Tengah': 'Sulawesi',
    'Papua Barat Daya': 'Papua', 'Papua Selatan': 'Papua'
}
sites['pulau'] = sites['provinsi'].map(provinsi_to_pulau)

# Tujuan 2 & 3: Pengelompokan 5 Kelas Kawasan Utama
def map_kelas_kawasan(st):
    st_low = str(st).lower()
    if 'peatland' in st_low: return 'Gambut / Rawan Asap'
    elif 'industrial' in st_low: return 'Kawasan Industri'
    elif 'traffic' in st_low or 'highway' in st_low: return 'Tepi Jalan / Transportasi'
    elif 'clean' in st_low or 'remote' in st_low or 'coastal / regional' in st_low: return 'Latar Bersih / Pesisir Alami'
    else: return 'Perkotaan (Urban)'

sites['kelas_kawasan'] = sites['site_type'].apply(map_kelas_kawasan)

# Tujuan 4: Kunci relasi eksplisit SENS-0nn <-> RS-XXX-nn
sensors['sensor_num'] = sensors['sensor_id'].str.extract(r'SENS-(\d+)')
sensors['hospital_id'] = 'RS-XXX-' + sensors['sensor_num']

master_station = pd.merge(
    sensors[['sensor_id', 'location_id', 'hospital_id', 'nama_sensor', 'model_sensor', 'tipe_pemasangan', 'status_operasional', 'tanggal_instalasi']],
    sites[['location_id', 'station_code', 'provinsi', 'pulau', 'kota_kabupaten', 'kecamatan', 'karakteristik_wilayah', 'site_type', 'kelas_kawasan', 'latitude_dd', 'longitude_dd', 'baseline_pm25', 'distance_to_road_m', 'traffic_intensity_per_h']],
    on='location_id', how='left'
)
master_station = pd.merge(
    master_station,
    hospitals[['hospital_id', 'nama_rumah_sakit', 'kelas_faskes', 'kapasitas_tempat_tidur']],
    on='hospital_id', how='left'
)
master_station['lat_long'] = master_station['latitude_dd'].astype(str) + ',' + master_station['longitude_dd'].astype(str)
print('Master stasiun + rumah sakit siap:', master_station.shape)

Master stasiun + rumah sakit siap: (27, 25)


## Tahap 2: Preprocessing, Data Cleaning, Konversi Satuan, & Audit Jaringan
- **Tujuan 1 (Validasi Data):** Menyaring hanya `qc_flag == 'VALID'` serta mengonversi satuan `ppm` ke $\mu g/m^3$ pada stasiun Banjarbaru, Medan, dan Pesawaran.
- **Tujuan 2 (Episode Kabut Asap):** Membersihkan sentinel (`-9999`, `9999`, `ERR`, nilai negatif) tanpa menghapus puncak kabut asap asli (>1.000 $\mu g/m^3$).
- **Tujuan 3 (Faktor Meteorologi):** Membersihkan suhu/kelembapan serta mendekomposisi arah angin ke vektor `sin/cos` sebelum *resampling* per jam.
- **Tujuan 5 (Kecukupan Jaringan):** Menghitung persentase baris `VALID` seluruh 27 stasiun dan mengeluarkan `SENS-027` (Muaro Jambi, 99,5% rusak) dari agregasi kualitas udara.
- **Dashboard:** Menyediakan data kualitas jaringan untuk Peta & Tabel Status Halaman 3.

In [3]:
BM_GAS = {'so2': 64.06, 'co': 28.01, 'o3': 48.00, 'no2': 46.01}
POLUTAN_COLS = ['pm25_ug_m3', 'pm10_ug_m3', 'so2_ug_m3', 'co_ug_m3', 'o3_ug_m3', 'no2_ug_m3']
SENTINEL_VALS = [-9999, 9999, 8888, 50000, 100000, -273.2]

aqms_files = sorted(glob.glob('*_AQMS.csv'))
clean_hourly_list = []
network_audit_list = []

for f in aqms_files:
    df = pd.read_csv(f, low_memory=False)
    sid = df['sensor_id'].dropna().iloc[0]
    total_rows = len(df)
    valid_rows = int((df['qc_flag'] == 'VALID').sum())
    pct_valid = round((valid_rows / total_rows) * 100, 2) if total_rows > 0 else 0.0
    
    # Audit jaringan (Tujuan 5)
    invalid_flags = df[df['qc_flag'] != 'VALID']['qc_flag']
    top_err = invalid_flags.mode()[0] if len(invalid_flags) > 0 else 'NONE'
    network_audit_list.append({
        'sensor_id': sid,
        'total_baris_mentah': total_rows,
        'baris_valid': valid_rows,
        'persen_valid': pct_valid,
        'persen_tidak_valid': round(100.0 - pct_valid, 2),
        'dominant_error_flag': top_err,
        'kategori_kualitas_data': (
            'Rusak / Defective (<10%)' if pct_valid < 10
            else ('Kritis / Rendah (<50%)' if pct_valid < 50
                  else ('Perlu Perhatian (50-90%)' if pct_valid < 90 else 'Baik (>=90%)'))
        )
    })
    
    # Filter VALID & drop sensor mati Muaro Jambi (SENS-027)
    df = df[df['qc_flag'] == 'VALID'].copy()
    if sid == 'SENS-027' or len(df) == 0:
        continue
        
    # Konversi satuan ppm -> ug/m3 (Banjarbaru, Medan, Pesawaran)
    if 'pm25_ppm' in df.columns:
        df['pm25_ug_m3'] = pd.to_numeric(df['pm25_ppm'], errors='coerce') * 1000.0
        df['pm10_ug_m3'] = pd.to_numeric(df['pm10_ppm'], errors='coerce') * 1000.0
        for gas, bm in BM_GAS.items():
            df[f'{gas}_ug_m3'] = pd.to_numeric(df[f'{gas}_ppm'], errors='coerce') * (bm / 24.45) * 1000.0
            
    # Pembersihan sentinel & batas fisik
    for col in POLUTAN_COLS:
        df[col] = pd.to_numeric(df[col], errors='coerce').replace([np.inf, -np.inf] + SENTINEL_VALS, np.nan)
        df.loc[(df[col] < 0) | (df[col] > 10000), col] = np.nan
        
    for col in ['temperature_c', 'humidity_pct', 'wind_speed_ms', 'wind_direction_deg']:
        df[col] = pd.to_numeric(df[col], errors='coerce').replace([np.inf, -np.inf] + SENTINEL_VALS, np.nan)
        
    df.loc[(df['temperature_c'] < -10) | (df['temperature_c'] > 55), 'temperature_c'] = np.nan
    df.loc[(df['humidity_pct'] < 0) | (df['humidity_pct'] > 100), 'humidity_pct'] = np.nan
    df.loc[(df['wind_speed_ms'] < 0) | (df['wind_speed_ms'] > 60), 'wind_speed_ms'] = np.nan
    df.loc[(df['wind_direction_deg'] < 0) | (df['wind_direction_deg'] > 360), 'wind_direction_deg'] = np.nan
    
    # Resample per jam terlebih dahulu (mengatasi stasiun resolusi 15-menit)
    df['timestamp_start'] = pd.to_datetime(df['timestamp_start'], errors='coerce')
    df = df.dropna(subset=['timestamp_start'])
    df['hour_ts'] = df['timestamp_start'].dt.floor('h')
    
    # Vektorisasi arah angin sirkular
    rad = np.deg2rad(df['wind_direction_deg'])
    df['wind_u'] = np.sin(rad)
    df['wind_v'] = np.cos(rad)
    
    agg_hourly = {c: 'mean' for c in POLUTAN_COLS + ['temperature_c', 'humidity_pct', 'wind_speed_ms', 'wind_u', 'wind_v']}
    df_hourly = df.groupby(['sensor_id', 'hour_ts'], as_index=False).agg(agg_hourly)
    df_hourly['wind_direction_deg'] = (np.rad2deg(np.arctan2(df_hourly['wind_u'], df_hourly['wind_v'])) + 360) % 360
    df_hourly = df_hourly.drop(columns=['wind_u', 'wind_v'])
    clean_hourly_list.append(df_hourly)

df_all_hourly = pd.concat(clean_hourly_list, ignore_index=True)
print('Sanity Check Nasional PM2.5 vs PM10:', round(df_all_hourly['pm25_ug_m3'].mean(), 2), 'vs', round(df_all_hourly['pm10_ug_m3'].mean(), 2))

Sanity Check Nasional PM2.5 vs PM10: 42.33 vs 61.76


## Tahap 3: EDA & Ekspor Tabel Profil Jam × Bulan
- **Tujuan 2 (Waktu Kritis):** Mengekstrak pola diurnal (jam sibuk pagi 06–08 & sore 17–19), perbandingan Hari Kerja vs Akhir Pekan, serta lonjakan musiman.
- **Dashboard:** Menyimpan `looker_2_profil_jam_bulan.csv` untuk grafik **Heatmap Jam × Bulan** di **Halaman 2**.

In [4]:
df_all_hourly['jam'] = df_all_hourly['hour_ts'].dt.hour
df_all_hourly['bulan_num'] = df_all_hourly['hour_ts'].dt.month
df_all_hourly['nama_bulan'] = df_all_hourly['hour_ts'].dt.strftime('%b')
df_all_hourly['tipe_hari'] = np.where(df_all_hourly['hour_ts'].dt.dayofweek >= 5, 'Akhir Pekan', 'Hari Kerja')

df_jam_bulan = (
    df_all_hourly.groupby(['sensor_id', 'bulan_num', 'nama_bulan', 'tipe_hari', 'jam'], as_index=False)
    .agg(
        pm25_rata2=('pm25_ug_m3', 'mean'),
        pm10_rata2=('pm10_ug_m3', 'mean'),
        so2_rata2=('so2_ug_m3', 'mean'),
        co_rata2=('co_ug_m3', 'mean'),
        o3_rata2=('o3_ug_m3', 'mean'),
        no2_rata2=('no2_ug_m3', 'mean'),
        suhu_rata2=('temperature_c', 'mean'),
        kelembapan_rata2=('humidity_pct', 'mean'),
        jumlah_jam_valid=('pm25_ug_m3', 'count')
    )
)
df_jam_bulan = pd.merge(df_jam_bulan, master_station, on='sensor_id', how='left')
df_jam_bulan.to_csv('looker_2_profil_jam_bulan.csv', index=False)
print('Tersimpan: looker_2_profil_jam_bulan.csv', df_jam_bulan.shape)

Tersimpan: looker_2_profil_jam_bulan.csv (14972, 38)


## Tahap 4: Agregasi Harian, Evaluasi Baku Mutu, & Analisis Dampak Kesehatan (Lag 0–7 Hari)
- **Tujuan 1 (Kepatuhan Baku Mutu):** Mengevaluasi pelanggaran terhadap **PP No. 22 Tahun 2021** (PM2.5 harian > 55 $\mu g/m^3$, PM10 > 75 $\mu g/m^3$), pedoman **WHO** (PM2.5 > 15 $\mu g/m^3$), serta kategori **ISPU** (Permen LHK No. 14/2020).
- **Tujuan 2 & 3 (Kabut Asap & Arah Angin):** Mengidentifikasi hari kabut asap (>150 $\mu g/m^3$) dan mengelompokkan arah angin ke 8 sektor kompas.
- **Tujuan 4 (Dampak Kesehatan & Kelompok Rentan):** Menghitung *lag* paparan PM2.5 (1–7 hari), *moving average* 7 hari, serta menormalisasi rawat inap **balita (<5 thn)** dan **lansia (>60 thn)** per 100 tempat tidur RS.
- **Dashboard:** Menyimpan `looker_1_master_stasiun_harian.csv` sebagai tabel utama untuk **Halaman 1, Halaman 2, dan Halaman 3** tanpa perlu *data blending*.

In [5]:
df_all_hourly['date'] = df_all_hourly['hour_ts'].dt.strftime('%Y-%m-%d')
rad_h = np.deg2rad(df_all_hourly['wind_direction_deg'])
df_all_hourly['wind_u'] = np.sin(rad_h)
df_all_hourly['wind_v'] = np.cos(rad_h)

df_daily = (
    df_all_hourly.groupby(['sensor_id', 'date'], as_index=False)
    .agg(
        jam_valid_harian=('pm25_ug_m3', 'count'),
        pm25_ug_m3=('pm25_ug_m3', 'mean'),
        pm25_max_jam=('pm25_ug_m3', 'max'),
        pm10_ug_m3=('pm10_ug_m3', 'mean'),
        so2_ug_m3=('so2_ug_m3', 'mean'),
        co_ug_m3=('co_ug_m3', 'mean'),
        o3_ug_m3=('o3_ug_m3', 'mean'),
        no2_ug_m3=('no2_ug_m3', 'mean'),
        temperature_c=('temperature_c', 'mean'),
        humidity_pct=('humidity_pct', 'mean'),
        wind_speed_ms=('wind_speed_ms', 'mean'),
        wind_u=('wind_u', 'mean'),
        wind_v=('wind_v', 'mean')
    )
)

# Syarat minimal 18 jam observasi valid per hari
df_daily = df_daily[df_daily['jam_valid_harian'] >= 18].copy()
df_daily['wind_direction_deg'] = (np.rad2deg(np.arctan2(df_daily['wind_u'], df_daily['wind_v'])) + 360) % 360
df_daily = df_daily.drop(columns=['wind_u', 'wind_v'])

def get_wind_sector(deg):
    if pd.isna(deg): return 'Tidak Diketahui'
    dirs = ['Utara (N)', 'Timur Laut (NE)', 'Timur (E)', 'Tenggara (SE)',
            'Selatan (S)', 'Barat Daya (SW)', 'Barat (W)', 'Barat Laut (NW)']
    return dirs[int((deg + 22.5) // 45) % 8]

df_daily['sektor_arah_angin'] = df_daily['wind_direction_deg'].apply(get_wind_sector)

# Kategori ISPU & Baku Mutu (Tujuan 1)
def kategori_ispu_pm25(val):
    if pd.isna(val): return 'Tidak Ada Data'
    elif val <= 15.5: return 'Baik'
    elif val <= 55.4: return 'Sedang'
    elif val <= 150.4: return 'Tidak Sehat'
    elif val <= 250.4: return 'Sangat Tidak Sehat'
    else: return 'Berbahaya'

df_daily['kategori_ispu_pm25'] = df_daily['pm25_ug_m3'].apply(kategori_ispu_pm25)
df_daily['melebihi_pp22_harian_pm25_55'] = (df_daily['pm25_ug_m3'] > 55).astype(int)
df_daily['melebihi_who_harian_pm25_15'] = (df_daily['pm25_ug_m3'] > 15).astype(int)
df_daily['melebihi_pp22_harian_pm10_75'] = (df_daily['pm10_ug_m3'] > 75).astype(int)
df_daily['melebihi_who_harian_pm10_45'] = (df_daily['pm10_ug_m3'] > 45).astype(int)
df_daily['status_hari_asap'] = np.where(df_daily['pm25_ug_m3'] > 150, 'Hari Kabut Asap (>150)', 'Hari Normal / Non-Asap')

# Lag 1-7 Hari & Moving Average 7 Hari (Tujuan 4)
df_daily = df_daily.sort_values(['sensor_id', 'date']).reset_index(drop=True)
for lag in range(1, 8):
    df_daily[f'pm25_lag_{lag}'] = df_daily.groupby('sensor_id')['pm25_ug_m3'].shift(lag)
df_daily['pm25_ma7'] = df_daily.groupby('sensor_id')['pm25_ug_m3'].transform(lambda s: s.rolling(7, min_periods=4).mean())

# Gabungkan dengan Master Stasiun & Kesehatan Harian
df_master_daily = pd.merge(df_daily, master_station, on='sensor_id', how='left')
health['date'] = pd.to_datetime(health['date']).dt.strftime('%Y-%m-%d')
df_master_daily = pd.merge(
    df_master_daily,
    health[['date', 'hospital_id', 'dow', 'resp_admissions_under5', 'resp_admissions_over60', 'asthma_emergency_cases', 'copd_admissions', 'resp', 'cvd', 'death']],
    on=['date', 'hospital_id'], how='left'
)

# Normalisasi kasus kesehatan per 100 tempat tidur RS
bed_factor = df_master_daily['kapasitas_tempat_tidur'] / 100.0
df_master_daily['balita_per_100_bed'] = df_master_daily['resp_admissions_under5'] / bed_factor
df_master_daily['lansia_per_100_bed'] = df_master_daily['resp_admissions_over60'] / bed_factor
df_master_daily['resp_total_per_100_bed'] = df_master_daily['resp'] / bed_factor
df_master_daily['cvd_per_100_bed'] = df_master_daily['cvd'] / bed_factor
df_master_daily['rasio_kerentanan_balita_vs_lansia'] = df_master_daily['resp_admissions_under5'] / df_master_daily['resp_admissions_over60'].replace(0, np.nan)
df_master_daily['resp_ma7'] = df_master_daily.groupby('sensor_id')['resp'].transform(lambda s: s.rolling(7, min_periods=4).mean())

# Dimensi Kalender Looker Studio
dt_col = pd.to_datetime(df_master_daily['date'])
df_master_daily['tahun'] = dt_col.dt.year
df_master_daily['bulan_num'] = dt_col.dt.month
df_master_daily['tahun_bulan'] = dt_col.dt.strftime('%Y-%m')
df_master_daily['minggu_ke'] = dt_col.dt.to_period('W').apply(lambda r: r.start_time.strftime('%Y-%m-%d'))
df_master_daily['tipe_hari'] = np.where(dt_col.dt.dayofweek >= 5, 'Akhir Pekan', 'Hari Kerja')

df_master_daily.to_csv('looker_1_master_stasiun_harian.csv', index=False)
print('Tersimpan: looker_1_master_stasiun_harian.csv', df_master_daily.shape)

Tersimpan: looker_1_master_stasiun_harian.csv (24236, 72)


## Tahap 5: Audit Kelengkapan Jaringan Sensor & Korelasi Pemicu Dalam-Stasiun
- **Tujuan 1:** Menandai stasiun yang melampaui **Baku Mutu Tahunan PP 22/2021** (PM2.5 > 15 $\mu g/m^3$).
- **Tujuan 3:** Menghitung korelasi **Spearman dalam-stasiun** antara PM2.5 dengan suhu, kelembapan, dan kecepatan angin.
- **Tujuan 5:** Menyusun tabel evaluasi seluruh 27 stasiun untuk mengidentifikasi titik pemantauan yang rusak/buta.
- **Dashboard:** Menyimpan `looker_3_audit_jaringan_sensor.csv` untuk **Halaman 2** (Peringkat Pemicu) dan **Halaman 3** (Peta & Tabel Status Jaringan).

In [6]:
df_audit = pd.DataFrame(network_audit_list)
station_metrics = []
for sid, grp in df_master_daily.groupby('sensor_id'):
    pm25_mean = grp['pm25_ug_m3'].mean()
    pm10_mean = grp['pm10_ug_m3'].mean()
    station_metrics.append({
        'sensor_id': sid,
        'hari_valid_terukur': len(grp),
        'pm25_rata2_periode': round(pm25_mean, 2),
        'pm10_rata2_periode': round(pm10_mean, 2),
        'melebihi_baku_mutu_tahunan_pp22_15': int(pm25_mean > 15),
        'melebihi_baku_mutu_tahunan_pm10_40': int(pm10_mean > 40),
        'total_hari_melebihi_55': int(grp['melebihi_pp22_harian_pm25_55'].sum()),
        'persen_hari_melebihi_55': round(grp['melebihi_pp22_harian_pm25_55'].mean() * 100, 2),
        'spearman_pm25_suhu': round(grp['pm25_ug_m3'].corr(grp['temperature_c'], method='spearman'), 3),
        'spearman_pm25_kelembapan': round(grp['pm25_ug_m3'].corr(grp['humidity_pct'], method='spearman'), 3),
        'spearman_pm25_kecepatan_angin': round(grp['pm25_ug_m3'].corr(grp['wind_speed_ms'], method='spearman'), 3),
        'pearson_pm25_resp_lag0': round(grp['pm25_ug_m3'].corr(grp['resp']), 3),
        'pearson_pm25_resp_lag1': round(grp['pm25_lag_1'].corr(grp['resp']), 3),
        'pearson_pm25_resp_lag2': round(grp['pm25_lag_2'].corr(grp['resp']), 3),
        'pearson_pm25_resp_lag3': round(grp['pm25_lag_3'].corr(grp['resp']), 3),
        'pearson_pm25_balita_lag0': round(grp['pm25_ug_m3'].corr(grp['resp_admissions_under5']), 3),
    })

df_station_metrics = pd.DataFrame(station_metrics)
df_network_final = pd.merge(master_station, df_audit, on='sensor_id', how='left')
df_network_final = pd.merge(df_network_final, df_station_metrics, on='sensor_id', how='left')
df_network_final.to_csv('looker_3_audit_jaringan_sensor.csv', index=False)
print('Tersimpan: looker_3_audit_jaringan_sensor.csv', df_network_final.shape)

Tersimpan: looker_3_audit_jaringan_sensor.csv (27, 46)
